In [ ]:
# @title Install

# !pip uninstall -y tensorflow jax jaxlib

# !pip install pytorch-ignite
# !pip install ipynbname
# !pip install lightning
# !pip install torch_tb_profiler
# !pip install colormaps
# !pip install captum

In [ ]:
# @title Importing

from __future__ import annotations

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
from torch.profiler import schedule
import torchmetrics
from ignite.engine import Events
from ignite.metrics import Metric, Loss
from ignite.handlers import global_step_from_engine, EarlyStopping
from torch.optim.lr_scheduler import _LRScheduler, ReduceLROnPlateau, CosineAnnealingWarmRestarts, ChainedScheduler
from ignite.contrib.handlers import TensorboardLogger
from ignite.contrib.handlers.tensorboard_logger import *
from ignite.exceptions import NotComputableError

import lightning as L
import pytorch_lightning as pl
from pytorch_lightning.loggers import TensorBoardLogger
from lightning.pytorch.callbacks import ModelCheckpoint, ModelSummary
from lightning.pytorch.callbacks.early_stopping import EarlyStopping
from lightning.pytorch.profilers import SimpleProfiler, PyTorchProfiler
from lightning.pytorch.utilities.warnings import PossibleUserWarning

from torch.optim.lr_scheduler import SequentialLR

from captum.attr import Saliency, IntegratedGradients, Occlusion, GradientShap

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import FastICA

import statsmodels.api as sm
from statsmodels.multivariate.factor_rotation import promax

import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.cm import ScalarMappable
from matplotlib import colors
import matplotlib.colors as mcolors
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Patch
import colormaps as cmaps
import plotly.graph_objects as go

from tqdm.auto import tqdm

import seaborn as sns

from scipy import stats
from scipy.stats import norm
import scipy.io as sio
from scipy.interpolate import griddata

from joblib import Parallel, delayed

import numpy as np
import datetime
import time
import os
import sys
import glob
import ipynbname
import pickle
import json
import math
import warnings
import copy
import logging

logging.getLogger("lightning.pytorch").setLevel(logging.ERROR)
logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)

warnings.filterwarnings("ignore", category=PossibleUserWarning)
warnings.filterwarnings("ignore", message=".*does not have many workers.*")
warnings.filterwarnings("ignore", message=r".*Checkpoint directory .* exists and is not empty.*", category=UserWarning, module=r"lightning\.pytorch\.callbacks\.model_checkpoint")
seed = 1
torch.manual_seed(seed)
np.random.seed(seed)
rng = np.random.default_rng(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# @title Configuration

import copy

class DotDict(dict):
    def __init__(self, d=None):
        super().__init__()
        if d:
            for k, v in d.items():
                self[k] = self._wrap(v)

    def _wrap(self, value):
        if isinstance(value, dict):
            return DotDict(value)
        if isinstance(value, list):
            return [self._wrap(v) for v in value]
        return value

    def __getattr__(self, name):
        try:
            return self[name]
        except KeyError:
            raise AttributeError(name)

    def __setattr__(self, name, value):
        self[name] = self._wrap(value)

    def __delattr__(self, name):
        try:
            del self[name]
        except KeyError:
            raise AttributeError(name)

    def to_dict(self):
        out = {}
        for k, v in self.items():
            if isinstance(v, DotDict):
                out[k] = v.to_dict()
            elif isinstance(v, list):
                out[k] = [x.to_dict() if isinstance(x, DotDict) else x for x in v]
            else:
                out[k] = v
        return out

    def __deepcopy__(self, memo):
        return DotDict(copy.deepcopy(self.to_dict(), memo))


Conf = DotDict({
    "run_id": 1,
    "paths": {
        "data": "./data.mat",
        "logs": "./lightning_logs",
    },
    "session_idx": 15,
    "device": device,
    "training": {
        "batch_size": 128,
        "max_epoch": 200,
        "min_delta": 1e-5,
        "patience": 20,
    },
    "model_params": {
        "hidden_size_mean": 4,
        "n_heads_mean": 2,
        "n_layers_mean": 1,
        "nonlinearity_mean": "gelu",
        "dropout_mean": 0.2,
        "hidden_size_cov": 4,
        "n_heads_cov": 2,
        "n_layers_cov": 1,
        "nonlinearity_cov": "gelu",
        "dropout_cov": 0.2,
    },
    "optimization": {
        "optimizer_type": {
            "name": "Adam",
            "Adam": {
                "lr": 0.02,
                "weight_decay": 0,
            },
            "AdamW": {
                "lr": 0.01,
                "weight_decay": 0.0001,
            },
        },
        "scheduler_type": {
            "name": "CosineReduce",
            "CosineReduce": {
                "eta_min": 1e-4,
                "factor": 0.5,
                "patience": 10,
                "min_lr": 1e-3,
                "last_epoch": -1,
            },
            "Reduce": {
                "factor": 0.5,
                "patience":10,
                "min_lr": 1e-3,
            },
            "Cosine": {
                "T_mult": 1,
                "eta_min": 1e-5,
                "last_epoch": -1,
            }
        }
    }
})

In [ ]:
# @title Helpers

class Anscombe(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 2.0 * torch.sqrt(x + 3.0 / 8.0)

    def inv(self, x):
        return (x / 2.0) ** 2 - 3.0 / 8.0

class NeuralDataset(Dataset):
    def __init__(self, X_position, X_dense_task, X_sparse_task, Y, Y_mean, device):
        self.X_position = torch.tensor(X_position, dtype=torch.float32, device=device).transpose(1, 2)
        self.X_dense_task = torch.tensor(X_dense_task, dtype=torch.float32, device=device)

        X_sparse_task = torch.tensor(X_sparse_task, dtype=torch.float32, device=device)
        self.X_sparse_task = X_sparse_task.round().long()

        Y_tensor = torch.tensor(Y, dtype=torch.float32, device=device).transpose(1, 2)
        anscombe = Anscombe()
        Y_anscombed = anscombe.forward(Y_tensor)
        self.Y = Y_anscombed - Y_mean
        
    def __len__(self):
        return self.X_position.size(0)

    def __getitem__(self, idx):
        X = (self.X_position[idx], self.X_dense_task[idx], self.X_sparse_task[idx])
        y = self.Y[idx]
        return X, y

class NeuralDataset(Dataset):
    def __init__(self, X_position, X_d_beh, X_s_beh, Y, mean, device):
        self.X_position = torch.tensor(X_position, dtype=torch.float32, device=device).transpose(1, 2)
        self.X_d_beh = torch.tensor(X_d_beh, dtype=torch.float32, device=device)

        X_s_beh = torch.tensor(X_s_beh, dtype=torch.float32, device=device)
        self.X_s_beh = X_s_beh.round().long()

        Y_tensor = torch.tensor(Y, dtype=torch.float32, device=device).transpose(1, 2)
        anscombe = Anscombe()
        Y_anscombed = anscombe.forward(Y_tensor)
        self.Y = Y_anscombed - mean
        
    def __len__(self):
        return self.X_position.size(0)

    def __getitem__(self, idx):
        X = (self.X_position[idx], self.X_d_beh[idx], self.X_s_beh[idx])
        y = self.Y[idx]
        return X, y

class Anscombe(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 2.0 * torch.sqrt(x + 3.0 / 8.0)

    def inv(self, x):
        return (x / 2.0) ** 2 - 3.0 / 8.0

class MetricHistory(L.Callback):
    def __init__(self):
        super().__init__()
        self.history = {
            "epoch": [],
            "train_loss_epoch": [],
            "valid_loss_epoch": [],
            "train_correlation": [],
            "valid_correlation": [],
        }

    def on_validation_epoch_end(self, trainer, pl_module):
        metrics = trainer.callback_metrics

        self.history["epoch"].append(int(trainer.current_epoch))
        self.history["train_loss_epoch"].append(
            float(metrics["train_loss_epoch"].detach().cpu()) if "train_loss_epoch" in metrics else np.nan
        )
        self.history["valid_loss_epoch"].append(
            float(metrics["valid_loss_epoch"].detach().cpu()) if "valid_loss_epoch" in metrics else np.nan
        )
        self.history["train_correlation"].append(
            float(metrics["train_correlation"].detach().cpu()) if "train_correlation" in metrics else np.nan
        )
        self.history["valid_correlation"].append(
            float(metrics["valid_correlation"].detach().cpu()) if "valid_correlation" in metrics else np.nan
        )

class CosineAnnealingWithPlateauReduction(_LRScheduler):
    def __init__(self, optimizer, steps_per_epoch, eta_min, factor, patience, min_lr, last_epoch):
        self.steps_per_epoch = steps_per_epoch
        self.eta_min = eta_min
        self.factor = factor
        self.patience = patience
        self.min_lr = min_lr

        self.max_lr = optimizer.param_groups[0]['lr']

        self.best_valid_loss = float('inf')
        self.epochs_without_improvement = 0

        super().__init__(optimizer, last_epoch)

    def get_lr(self):
        step_in_epoch = self.last_epoch % self.steps_per_epoch

        cosine_decay = 0.5 * (1 + math.cos(math.pi * step_in_epoch / self.steps_per_epoch))
        lr = self.eta_min + (self.max_lr - self.eta_min) * cosine_decay

        return [lr for _ in self.base_lrs]

    def step_on_plateau(self, valid_loss):

        if valid_loss < self.best_valid_loss:
            self.best_valid_loss = valid_loss
            self.epochs_without_improvement = 0
        else:
            self.epochs_without_improvement += 1

            if self.epochs_without_improvement >= self.patience:
                old_max_lr = self.max_lr
                self.max_lr = max(self.max_lr * self.factor, self.min_lr)
                self.epochs_without_improvement = 0  # Reset counter

class CorrelationCoefficient(torchmetrics.Metric):
    def __init__(self, dist_sync_on_step=False):
        super().__init__(dist_sync_on_step=dist_sync_on_step)
        self.add_state("sum_xy", default=torch.tensor(0.0), dist_reduce_fx="sum")
        self.add_state("sum_x", default=torch.tensor(0.0), dist_reduce_fx="sum")
        self.add_state("sum_y", default=torch.tensor(0.0), dist_reduce_fx="sum")
        self.add_state("sum_x2", default=torch.tensor(0.0), dist_reduce_fx="sum")
        self.add_state("sum_y2", default=torch.tensor(0.0), dist_reduce_fx="sum")
        self.add_state("n", default=torch.tensor(0), dist_reduce_fx="sum")

    def update(self, y_pred: torch.Tensor, y: torch.Tensor):
        y_pred_flat = y_pred.flatten()
        y_flat = y.flatten()

        n = y_flat.numel()
        self.n += n
        self.sum_x += y_pred_flat.sum()
        self.sum_y += y_flat.sum()
        self.sum_xy += (y_pred_flat * y_flat).sum()
        self.sum_x2 += (y_pred_flat ** 2).sum()
        self.sum_y2 += (y_flat ** 2).sum()

    def compute(self):
        n = self.n.float()
        numerator = n * self.sum_xy - self.sum_x * self.sum_y
        denominator = torch.sqrt(
            (n * self.sum_x2 - self.sum_x ** 2) *
            (n * self.sum_y2 - self.sum_y ** 2)
        )
        return numerator / (denominator + 1e-8)

class MVNNLLLoss(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, out, y):
        mean, L = out
        
        if not torch.is_tensor(mean):
            mean = torch.stack(list(mean), dim=0)
        if not torch.is_tensor(y):
            y = torch.stack(list(y), dim=0)
        if not torch.is_tensor(L):
            L = torch.stack(list(L), dim=0)

        B = mean.shape[0]
        mean = mean.reshape(B, -1)
        y = y.reshape(B, -1)
        D = mean.shape[1]

        if L.dim() == 2:
            L = L.unsqueeze(0).expand(B, -1, -1)

        diff = (y - mean).unsqueeze(-1)
        sol = torch.linalg.solve_triangular(L, diff, upper=False).squeeze(-1)
        maha = (sol * sol).sum(dim=-1)
        logdet = 2.0 * torch.log(torch.diagonal(L, dim1=-2, dim2=-1)).sum(dim=-1)
        log_prob = -0.5 * (D * math.log(2.0 * math.pi) + logdet + maha)
        loss = (-log_prob / D).mean()
        
        return loss

def build_lit_model(Conf, mean_model, cov_model, fold_idx, steps_per_epoch, seq_size_cov):
    run_name = f"{mean_model}-mean-{cov_model}-cov"
    ckpt_dir = f"{Conf.paths.logs}/{run_name}/fold_{fold_idx}"

    logger = TensorBoardLogger(
        save_dir=Conf.paths.logs,
        name=run_name,
    )
    checkpoint_callback = ModelCheckpoint(
        dirpath=ckpt_dir,
        filename='model-{epoch:02d}-{valid_loss_epoch:.2f}',
        monitor='valid_loss_epoch',
        mode='min',
        save_top_k=3,
        save_last=True,
        verbose=False,
    )
    early_stop_callback = EarlyStopping(
        monitor='valid_loss_epoch',
        min_delta=Conf.training.min_delta,
        patience=Conf.training.patience,
        verbose=False,
        mode='min',
    )
    metric_history_callback = MetricHistory()

    trainer = L.Trainer(
        max_epochs=Conf.training.max_epoch,
        accelerator=Conf.device.type,
        devices=1,
        num_sanity_val_steps=0,
        logger=logger,
        callbacks=[checkpoint_callback, early_stop_callback, metric_history_callback],
        enable_progress_bar=True,
        enable_model_summary=False,
        detect_anomaly=False,
        log_every_n_steps=1,
    )
    lit_model = LitModel(Conf, mean_model, cov_model, steps_per_epoch, seq_size_cov).to(Conf.device)
    trainer.metric_history_callback = metric_history_callback
    
    return trainer, lit_model


In [ ]:
# @title Model

class Model(nn.Module):
    def __init__(self, Conf):
        super().__init__()
        self.seq_size = Conf.data_size.seq_size
        self.position_size = Conf.data_size.position_size
        self.dense_size = Conf.data_size.dense_size
        self.sparse_size = Conf.data_size.sparse_size
        self.output_size = Conf.data_size.output_size
        self.device = Conf.device


class MeanModel(Model):
    def __init__(self, Conf):
        super().__init__(Conf)
        self.input_size = self.sparse_size + self.dense_size + self.position_size
        self.nonlinearity = Conf.model_params.nonlinearity_mean
        self.hidden_size = Conf.model_params.hidden_size_mean
        self.n_heads = Conf.model_params.n_heads_mean
        self.n_layers = Conf.model_params.n_layers_mean
        self.dropout = Conf.model_params.dropout_mean


class BaselineMeanModel(MeanModel):
    def __init__(self, Conf):
        super().__init__(Conf)  
        self.theta = nn.Parameter(torch.empty(self.seq_size, self.output_size, device=self.device))
        
        with torch.no_grad():
            nn.init.normal_(self.theta, mean=0, std=1e-1)
            
    def forward(self, x):
        batch_size = x[0].size(0)
        mean = self.theta
        return mean.unsqueeze(0).expand(batch_size, -1, -1)

class Time2Vec(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.w0 = nn.Parameter(torch.randn(1))
        self.b0 = nn.Parameter(torch.randn(1))
        self.w = nn.Parameter(torch.randn(hidden_size - 1))
        self.b = nn.Parameter(torch.randn(hidden_size - 1))

    def forward(self, t):
        t = t.unsqueeze(-1)
        linear = self.w0 * t + self.b0
        periodic = torch.sin(self.w * t + self.b)
        return torch.cat([linear, periodic], dim=-1)
        
class Time2VecPositionalEncoding(nn.Module):
    def __init__(self, hidden_size, max_seq_size=500):
        super().__init__()
        self.hidden_size = hidden_size
        self.max_seq_size = max_seq_size
        self.t2v = Time2Vec(hidden_size)

    def forward(self, seq_size, device):
        t = torch.arange(seq_size, device=device).float()
        enc = self.t2v(t)
        return enc.unsqueeze(0)

        
class ConditionalMeanModel(MeanModel):
    def __init__(self, Conf):
        super().__init__(Conf)
        self.sparse_embeds = nn.ModuleList([
            nn.Embedding(2, 1) for _ in range(self.sparse_size)
        ])
        
        self.dense_proj = nn.Linear(self.dense_size, self.dense_size)
        self.position_proj = nn.Linear(self.position_size, self.position_size)
        self.input_proj = nn.Linear(self.input_size, self.hidden_size)
        
        self.pos_encoding = Time2VecPositionalEncoding(self.hidden_size, self.seq_size)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=self.hidden_size,
            nhead=self.n_heads,
            dim_feedforward=self.hidden_size * 4,
            dropout=self.dropout,
            activation=self.nonlinearity,
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=self.n_layers)
        self.dropout_layer = nn.Dropout(self.dropout)
        
        self.head = nn.Linear(self.hidden_size, self.output_size)

    def forward(self, x):
        batch_size = x[0].size(0)
        x_position, x_d_beh, x_s_beh = x
        
        sparse = torch.stack([self.sparse_embeds[i](x_s_beh[:, i]) for i in range(x_s_beh.size(1))], dim=1)
        sparse = sparse.squeeze(-1)
        sparse_expand = sparse.unsqueeze(1).expand(-1, self.seq_size, -1)
        dense = self.dense_proj(x_d_beh).unsqueeze(1)
        dense_expand = dense.expand(-1, self.seq_size, -1)
        position = self.position_proj(x_position)
        stacked = torch.cat([sparse_expand, dense_expand, position], dim=2)
        stacked = self.input_proj(stacked)
        
        pos_enc = self.pos_encoding(self.seq_size, self.device)
        combined = stacked + pos_enc
        transformed = self.transformer_encoder(combined)
        transformed = self.dropout_layer(transformed)
        
        mean = self.head(transformed)
        return mean
             
class CovModel(Model):
    def __init__(self, Conf, seq_size_cov):
        super().__init__(Conf)
        self.input_size = self.sparse_size + self.dense_size
        self.seq_size_cov = seq_size_cov
        self.hidden_size = Conf.model_params.hidden_size_cov
        self.nonlinearity = Conf.model_params.nonlinearity_cov
        self.n_heads = Conf.model_params.n_heads_cov
        self.n_layers = Conf.model_params.n_layers_cov
        self.dropout = Conf.model_params.dropout_cov
        
        self.register_buffer("I", torch.eye(self.seq_size * self.output_size, dtype=torch.float32, device=self.device))
        self.length_scales = nn.Parameter(torch.empty(self.hidden_size, device=self.device))
        self.noise = nn.Parameter(torch.empty(self.seq_size, self.output_size, device=self.device))

        with torch.no_grad():
            ls0 = math.log(math.expm1(0.5))
            self.length_scales.fill_(ls0)
            nn.init.normal_(self.length_scales, mean=ls0, std=1)
            nn.init.normal_(self.noise, mean=-20.0, std=1)

    def kernel(self, fun, n_samples1, n_samples2, l):
        i_grid = torch.arange(n_samples1, dtype=torch.float32, device=l.device).view(-1, 1)
        j_grid = torch.arange(n_samples2, dtype=torch.float32, device=l.device).view(1, -1)
        return fun(i_grid, j_grid, l)

    def squared_exponential_kernel(self, x1, x2, l):
        distances = (x2 - x1).unsqueeze(0)
        frac = distances / l.unsqueeze(-1).unsqueeze(-1)
        return torch.exp(-0.5 * (frac**2))

    def interp_lambda(self, lambda_matrix):
        if self.seq_size_cov == self.seq_size:
            return lambda_matrix
    
        if self.seq_size_cov == 1:
            return lambda_matrix.expand(self.seq_size, -1, -1)
    
        lambda_matrix = lambda_matrix.permute(1, 2, 0).reshape(1, self.output_size * self.hidden_size, self.seq_size_cov)
        lambda_matrix = F.interpolate(lambda_matrix, size=self.seq_size, mode="linear", align_corners=True)
        lambda_matrix = lambda_matrix.view(self.output_size, self.hidden_size, self.seq_size).permute(2, 0, 1)
        
        return lambda_matrix.contiguous()
    
    def _build_covariance_matrix(self, lambda_batch):
        if self.seq_size_cov == self.seq_size:
            Lambda = lambda_batch
        elif self.seq_size_cov == 1:
            lambda_batch = lambda_batch.expand(-1, self.seq_size, -1, -1)
        else:
            batch_size = lambda_batch.size(0)
            lambda_batch = lambda_batch.permute(0, 2, 3, 1).reshape(batch_size, self.output_size * self.hidden_size, self.seq_size_cov)
            lambda_batch = F.interpolate(lambda_batch, size=self.seq_size, mode="linear", align_corners=True)
            lambda_batch = lambda_batch.view(batch_size, self.output_size, self.hidden_size, self.seq_size).permute(0, 3, 1, 2).contiguous()
    
        l = F.softplus(self.length_scales)
        K = self.kernel(self.squared_exponential_kernel, self.seq_size, self.seq_size, l=l)
    
        blocks = torch.einsum('btok,kts,bsuk->btosu', lambda_batch, K, lambda_batch)
        cov = blocks.reshape(
            lambda_batch.size(0),
            self.seq_size * self.output_size,
            self.seq_size * self.output_size
        )
    
        cov = cov + self.I.unsqueeze(0)
        noise_diag = torch.sigmoid(self.noise.flatten())
        cov.diagonal(dim1=-2, dim2=-1).add_(noise_diag)
    
        L = torch.linalg.cholesky(cov)
        return L

class IdentityCovModel(CovModel):
    def __init__(self, Conf, seq_size_cov):
        super().__init__(Conf, seq_size_cov)
        
    def forward(self, x):
        batch_size = x[0].size(0)
        L = self.I.unsqueeze(0).expand(batch_size, -1, -1)
        return L
            
class SharedCovModel(CovModel):
    def __init__(self, Conf, seq_size_cov):
        super().__init__(Conf, seq_size_cov)
        self.lambda_matrix = nn.Parameter(torch.empty(self.seq_size_cov, self.output_size, self.hidden_size, device=self.device))

        with torch.no_grad():
            nn.init.normal_(self.lambda_matrix, mean=0.0, std=1e-1)

    def forward(self, x):
        batch_size = x[0].size(0)
        L = self._build_covariance_matrix(self.lambda_matrix.unsqueeze(0)).squeeze(0)
        L = L.unsqueeze(0).expand(batch_size, -1, -1)
        return L

class ConditionalCovModel(CovModel):
    def __init__(self, Conf, seq_size_cov):
        super().__init__(Conf, seq_size_cov)        
        self.sparse_embeds = nn.ModuleList([
            nn.Embedding(2, 1) for _ in range(self.sparse_size)
        ])

        self.dense_proj = nn.Linear(self.dense_size, self.dense_size)
        self.input_proj = nn.Linear(self.input_size, self.hidden_size)
        self.pos_encoding = Time2VecPositionalEncoding(self.hidden_size, self.seq_size_cov)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=self.hidden_size,
            nhead=self.n_heads,
            dim_feedforward=self.hidden_size * 4,
            dropout=self.dropout,
            activation=self.nonlinearity,
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=self.n_layers)
        self.dropout = nn.Dropout(self.dropout)
        
        self.head = nn.Linear(self.hidden_size, self.output_size * self.hidden_size)

    def forward(self, x):
        _, x_d_beh, x_s_beh = x
        
        sparse = torch.stack(
            [self.sparse_embeds[i](x_s_beh[:, i]) for i in range(x_s_beh.size(1))],
            dim=1
        )
        sparse = sparse.squeeze(-1)
        sparse_expand = sparse.unsqueeze(1).expand(-1, self.seq_size_cov, -1)

        dense = self.dense_proj(x_d_beh).unsqueeze(1)
        dense_expand = dense.expand(-1, self.seq_size_cov, -1)

        stacked = torch.cat([sparse_expand, dense_expand], dim=2)
        stacked = self.input_proj(stacked)
        pos_enc = self.pos_encoding(self.seq_size_cov, self.device)
        combined = stacked + pos_enc

        transformed = self.transformer_encoder(combined)
        transformed = self.dropout(transformed)
        
        lambda_batch = self.head(transformed).view(
            transformed.size(0), self.seq_size_cov, self.output_size, self.hidden_size
        )
    
        return self._build_covariance_matrix(lambda_batch)


class FullModel(Model):
    def __init__(self, Conf, mean_model, cov_model, seq_size_cov):
        super().__init__(Conf)
        if mean_model == 'baseline':
            self.mean_model = BaselineMeanModel(Conf)
        elif mean_model == 'conditional':
            self.mean_model = ConditionalMeanModel(Conf)

        if cov_model == 'identity':
            self.cov_model = IdentityCovModel(Conf, seq_size_cov)
        elif cov_model == 'shared':
            self.cov_model = SharedCovModel(Conf, seq_size_cov)
        elif cov_model == 'conditional':
            self.cov_model = ConditionalCovModel(Conf, seq_size_cov)
    
    def predict(self, mean, cov, y, variant='mean', **kwargs):
        if isinstance(variant, int):
            return self.__predict_by_unit_at_same_time(mean, cov, y, variant)
        elif variant == 'past':
            return self.__predict_by_past(mean, cov, y, **kwargs)
        elif variant == 'others':
            return self.__predict_by_others(mean, cov, y)
        elif variant == 'past_and_others':
            return self.__predict_by_past_and_others(mean, cov, y, **kwargs)
        else:
            raise ValueError(f'Variant is not supported: {variant}')
    
    def __predict_by_past(self, mean, cov, y, max_t_past_prediction=None):
        y_flat = y.flatten()
        mean_flat = mean.flatten()
        conditioned_mean = mean.clone().reshape((-1, self.output_size))
    
        for t in range(1, conditioned_mean.shape[0]):
            conditioned_indices = torch.arange(t * self.output_size, device=mean.device).reshape(t, self.output_size).t()
            product_indices = torch.stack((
                conditioned_indices.flatten().repeat_interleave(t).reshape(self.output_size, t, t),
                conditioned_indices.unsqueeze(1).repeat(1, t, 1)
            ), dim=-1)
    
            row_indices = torch.arange(t * self.output_size, (t + 1) * self.output_size, device=mean.device)
            cross_cov = cov[row_indices.unsqueeze(-1), conditioned_indices].unsqueeze(1)
            partial_cov = cov[product_indices[..., 0], product_indices[..., 1]].reshape(self.output_size, t, t)
            diff = (y_flat[conditioned_indices] - mean_flat[conditioned_indices]).unsqueeze(-1)
    
            conditioned_mean[t] += torch.matmul(torch.matmul(cross_cov, torch.linalg.inv(partial_cov)), diff).reshape(self.output_size)
    
        return conditioned_mean
    
    def __predict_by_others(self, mean, cov, y):
        nr_others = self.output_size - 1
        y_flat = y.flatten()
        mean_flat = mean.flatten()
        conditioned_mean = mean.clone().reshape((-1, self.output_size))
    
        for t in range(conditioned_mean.shape[0]):
            time_indices = torch.arange(t * self.output_size, (t + 1) * self.output_size, device=mean.device)
            conditioned_indices = time_indices.unsqueeze(0).repeat(self.output_size, 1)
            conditioned_indices = conditioned_indices[~torch.eye(self.output_size, dtype=torch.bool, device=mean.device)].reshape(self.output_size, nr_others)
    
            product_indices = torch.stack((
                conditioned_indices.flatten().repeat_interleave(nr_others).reshape(self.output_size, nr_others, nr_others),
                conditioned_indices.unsqueeze(1).repeat(1, nr_others, 1)
            ), dim=-1)
    
            cross_cov = cov[time_indices.unsqueeze(-1), conditioned_indices].unsqueeze(1)
            partial_cov = cov[product_indices[..., 0], product_indices[..., 1]].reshape(self.output_size, nr_others, nr_others)
            diff = (y_flat[conditioned_indices] - mean_flat[conditioned_indices]).unsqueeze(-1)
    
            conditioned_mean[t] += torch.matmul(torch.matmul(cross_cov, torch.linalg.inv(partial_cov)), diff).reshape(self.output_size)
    
        return conditioned_mean
    
    def __predict_by_past_and_others(self, mean, cov, y, max_t_past_prediction=None):
        nr_others = self.output_size - 1
        y_flat = y.flatten()
        mean_flat = mean.flatten()
        conditioned_mean = mean.clone().reshape((-1, self.output_size))
    
        for t in range(conditioned_mean.shape[0]):
            nr_past_and_others = t + nr_others
    
            past_indices = torch.arange(t * self.output_size, device=mean.device).reshape(t, self.output_size).t()
            time_indices = torch.arange(t * self.output_size, (t + 1) * self.output_size, device=mean.device)
            others_indices = time_indices.unsqueeze(0).repeat(self.output_size, 1)
            others_indices = others_indices[~torch.eye(self.output_size, dtype=torch.bool, device=mean.device)].reshape(self.output_size, nr_others)
    
            conditioned_indices = torch.cat((past_indices, others_indices), dim=-1)
    
            product_indices = torch.stack((
                conditioned_indices.flatten().repeat_interleave(nr_past_and_others).reshape(self.output_size, nr_past_and_others, nr_past_and_others),
                conditioned_indices.unsqueeze(1).repeat(1, nr_past_and_others, 1)
            ), dim=-1)
    
            cross_cov = cov[time_indices.unsqueeze(-1), conditioned_indices].unsqueeze(1)
            partial_cov = cov[product_indices[..., 0], product_indices[..., 1]].reshape(self.output_size, nr_past_and_others, nr_past_and_others)
            diff = (y_flat[conditioned_indices] - mean_flat[conditioned_indices]).unsqueeze(-1)
    
            conditioned_mean[t] += torch.matmul(torch.matmul(cross_cov, torch.linalg.inv(partial_cov)), diff).reshape(self.output_size)
    
        return conditioned_mean

    def __predict_by_unit_at_same_time(self, mean, cov, y, unit_idx):
        device = mean.device
        y_flat = y.flatten()
        mean_flat = mean.flatten()
        conditioned_mean = mean.clone().reshape((-1, self.output_size))
        jitter = 1e-12
    
        for t in range(conditioned_mean.shape[0]):
            time_indices = t * self.output_size + torch.arange(self.output_size, device=device)
            cond_idx = t * self.output_size + int(unit_idx)

            cross_cov = cov[time_indices, cond_idx]
            var_cond = cov[cond_idx, cond_idx] + jitter
            diff = y_flat[cond_idx] - mean_flat[cond_idx]
    
            conditioned_mean[t] = mean[t] + (cross_cov / var_cond) * diff
    
        return conditioned_mean

    def forward(self, x, y, variant):
        mean = self.mean_model(x)
        L = self.cov_model(x)
        if variant == 'mean':
            return mean, L
        else:
            cov = L @ L.transpose(-1, -2)
            mean = torch.stack(
                [self.predict(mean_sample, cov_sample, y_sample, variant=variant) for mean_sample, cov_sample, y_sample in zip(mean, cov, y)],
                dim=0
            )
            return mean, L

def eval_mean_model(module):
    if not any(p.requires_grad for p in module.full_model.mean_model.parameters()):
        module.full_model.mean_model.eval()

class LitModel(L.LightningModule):
    def __init__(self, Conf, mean_model, cov_model, steps_per_epoch, seq_size_cov):
        super().__init__()
        self.save_hyperparameters(ignore=["optimizer_type", "scheduler_type", "data_size", "model_params"])        
        self.steps_per_epoch = steps_per_epoch
        self.optimizer_name = Conf.optimization.optimizer_type.name
        self.optimizer_params = Conf.optimization.optimizer_type[self.optimizer_name]
        self.scheduler_name = Conf.optimization.scheduler_type.name
        self.scheduler_params = Conf.optimization.scheduler_type[self.scheduler_name]

        self.full_model = FullModel(Conf, mean_model, cov_model, seq_size_cov)
        self.mvn_nll_loss = MVNNLLLoss()
        self.train_corr = CorrelationCoefficient()
        self.valid_corr = CorrelationCoefficient()
        self.anscombe = Anscombe()
     
    def forward(self, x, y, variant='mean'):
        return self.full_model(x, y, variant)

    def training_step(self, batch, batch_idx):
        x, y = batch
        eval_mean_model(self)
        out = self(x, y, 'mean')
        loss = self.mvn_nll_loss(out, y)
        self.train_corr.update(out[0], y)

        self.log("train_loss_step", loss, on_step=True, on_epoch=False, logger=True)
        self.log("train_loss_epoch", loss, on_step=False, on_epoch=True, logger=True, prog_bar=True)
        self.log("learning_rate_step", self.trainer.optimizers[0].param_groups[0]['lr'],
                on_step=True, on_epoch=False, logger=True)
        return loss

    def on_train_epoch_end(self):
        corr = self.train_corr.compute()
        self.log('train_correlation', corr, logger=True, prog_bar=True)
        self.train_corr.reset()

    def validation_step(self, batch, batch_idx):
        x, y = batch
        eval_mean_model(self)
        out = self(x, y, 'mean')
        loss = self.mvn_nll_loss(out, y)
        self.valid_corr.update(out[0], y)

        self.log("valid_loss_epoch", loss, on_step=False, on_epoch=True, logger=True, prog_bar=True)

    def on_validation_epoch_end(self):
        loss = self.trainer.callback_metrics.get('valid_loss_epoch')
        if loss is not None and self.scheduler_name == 'CosineReduce':
            self.scheduler.step_on_plateau(loss.item())

        corr = self.valid_corr.compute()
        self.log('valid_correlation', corr, logger=True, prog_bar=True)
        self.valid_corr.reset()

    def configure_optimizers(self):
        params = [p for p in self.parameters() if p.requires_grad]

        if self.optimizer_name == "Adam":
            optimizer = torch.optim.Adam(
                params,
                lr=self.optimizer_params.lr,
                weight_decay=self.optimizer_params.weight_decay
            )
        elif self.optimizer_name == "AdamW":
            optimizer = torch.optim.AdamW(
                params,
                lr=self.optimizer_params.lr,
                weight_decay=self.optimizer_params.weight_decay
            )

        if self.scheduler_name == None:
            return optimizer

        elif self.scheduler_name == 'CosineReduce':
            self.scheduler = CosineAnnealingWithPlateauReduction(
                optimizer,
                steps_per_epoch=self.steps_per_epoch,
                eta_min=self.scheduler_params.eta_min,
                factor=self.scheduler_params.factor,
                patience=self.scheduler_params.patience,
                min_lr=self.scheduler_params.min_lr,
                last_epoch=self.scheduler_params.last_epoch
            )
            return {
                'optimizer': optimizer,
                'lr_scheduler': {
                    'scheduler': self.scheduler,
                    'interval': 'step',
                    'frequency': 1
                }
            }

        elif self.scheduler_name == 'Reduce':
            scheduler = ReduceLROnPlateau(
                optimizer,
                mode='min',
                factor=self.scheduler_params.factor,
                patience=self.scheduler_params.patience,
                min_lr=self.scheduler_params.min_lr
            )
            return {
                'optimizer': optimizer,
                'lr_scheduler': {
                    'scheduler': scheduler,
                    'monitor': 'valid_loss_epoch',
                    'interval': 'epoch',
                    'frequency': 1
                }
            }

        elif self.scheduler_name == 'Cosine':
            scheduler = CosineAnnealingWarmRestarts(
                optimizer,
                T_0=self.steps_per_epoch,
                T_mult=self.scheduler_params.T_mult,
                eta_min=self.scheduler_params.eta_min,
                last_epoch=self.scheduler_params.last_epoch
            )
            return {
                'optimizer': optimizer,
                'lr_scheduler': {
                    'scheduler': scheduler,
                    'interval': 'step',
                    'frequency': 1
                }
            }


In [ ]:
data = sio.loadmat(Conf.paths.data, squeeze_me=True, struct_as_record=False)

session_name = data['session_names'][Conf.session_idx]
spikes = data['spikes'][Conf.session_idx]
position_vars = data['position_vars'][Conf.session_idx]
position_var_names = data['position_var_names']
task_vars = data['task_vars'][Conf.session_idx]
task_var_names = data['task_var_names']
events = data['events'][Conf.session_idx]
event_names = data['event_names']
unit_names = data['unit_names'][Conf.session_idx]
channel_names = data['channel_names'][Conf.session_idx]
unit_types = data['unit_types'][Conf.session_idx]
bin_size = data['bin_size']
bin_times = data['bin_times']

spikes  = np.transpose(spikes, (1, 0, 2)) * 0.2
position_vars  = np.transpose(position_vars, (1, 0, 2))
task_vars = np.transpose(task_vars, (1, 0))

dense = [0, 1, 2, 3]
sparse = [4, 5, 6]

print(f"session_idx = {Conf.session_idx}, session_name = {session_name}")
print(f'task_var_names = {task_var_names}')
print(f'spikes.shape = {spikes.shape}')

In [ ]:
units_to_remove = [14, 15, 20, 82, 91, 175]

units_name_delete = [ 6,  10,  12,  13,  14,  15,  20,  22,  23,  27,
        32,  33,  40,  41,  42,  43,  48,  49,  50,  51,  58,  59,
        61,  64,  65,  70,  71,  73,  76,  77,  78,  82,  83,  86,
        91,  92, 101, 102, 107, 108, 111, 113, 114, 119, 123, 128,
       129, 132, 133, 134, 135, 136, 137, 142, 143, 147, 148, 149, 153,
       154, 156, 157, 158, 160, 161, 164, 175, 177, 178, 181,
       182, 188]
units_mask_delete = np.isin(unit_names, units_name_delete)
unit_names = unit_names[~units_mask_delete]
spikes = spikes[:, ~units_mask_delete]

In [ ]:
spike_mask_finite = np.isfinite(spikes).all(axis=(1, 2))
position_vars_mask_finite = np.isfinite(position_vars).all(axis=(1, 2))
task_vars_mask_finite = np.isfinite(task_vars).all(axis=1)
mask_finite = spike_mask_finite & position_vars_mask_finite & task_vars_mask_finite
print("deleted infinite spike:", (~spike_mask_finite).sum())
print("deleted infinite position_vars:", (~position_vars_mask_finite).sum())
print("deleted infinite task_vars:", (~task_vars_mask_finite).sum())

position_vars_mask_zero = (position_vars == 0).all(axis=(1, 2))
print("deleted zero position_vars:", (position_vars_mask_zero).sum())

task_vars_mask_over = np.zeros(task_vars.shape[0], dtype=bool)
for task_var_name in ['tunp', 'tslp']:
    task_var_idx = np.where(task_var_names == task_var_name)[0][0]
    task_var_mask_over = task_vars[:, task_var_idx] > 100
    print(f"deleted over {task_var_name}:", task_var_mask_over.sum())
    task_vars_mask_over |= task_var_mask_over

trials_mask_keep = mask_finite & ~position_vars_mask_zero & ~task_vars_mask_over

spikes = spikes[trials_mask_keep]
position_vars = position_vars[trials_mask_keep]
task_vars = task_vars[trials_mask_keep]

for task_var_name in ['tunp', 'tslp']:
    task_var_idx = np.where(task_var_names == task_var_name)[0][0]
    task_vars[:, task_var_idx] = np.log(task_vars[:, task_var_idx])

scaler = MinMaxScaler(feature_range=(0, 1))
task_vars = scaler.fit_transform(task_vars)

n_trials, n_units , n_bins = spikes.shape
n_trials, n_position_vars, n_bins = position_vars.shape
n_trials, n_task_vars = task_vars.shape

Conf.data_size = DotDict()
Conf.data_size.seq_size = n_bins
Conf.data_size.position_size = n_position_vars
Conf.data_size.dense_size = len(dense)
Conf.data_size.sparse_size = len(sparse)
Conf.data_size.output_size = n_units

In [ ]:
X_dense_task = task_vars[:, dense]
X_sparse_task = task_vars[:, sparse]
X_position = position_vars
Y = spikes

train_loader_folds = []
valid_loader_folds = []
kf = KFold(n_splits=10, shuffle=True, random_state=seed)

for train_idx, valid_idx in kf.split(Y):
    X_position_train = X_position[train_idx]
    X_position_valid = X_position[valid_idx]

    X_dense_task_train = X_dense_task[train_idx]
    X_dense_task_valid = X_dense_task[valid_idx]

    X_sparse_task_train = X_sparse_task[train_idx]
    X_sparse_task_valid = X_sparse_task[valid_idx]

    Y_train = Y[train_idx]
    Y_valid = Y[valid_idx]

    position_Y_mean = X_position_train.mean(axis=0, keepdims=True)
    position_std = X_position_train.std(axis=0, keepdims=True)
    X_position_train = (X_position_train - position_Y_mean) / (position_std + 1e-8)
    X_position_valid = (X_position_valid - position_Y_mean) / (position_std + 1e-8)

    dense_task_Y_mean = X_dense_task_train.mean(axis=0, keepdims=True)
    dense_task_std = X_dense_task_train.std(axis=0, keepdims=True)
    X_dense_task_train = (X_dense_task_train - dense_task_Y_mean) / (dense_task_std + 1e-8)
    X_dense_task_valid = (X_dense_task_valid - dense_task_Y_mean) / (dense_task_std + 1e-8)

    Y_train_tensor = torch.tensor(Y_train, dtype=torch.float32, device=device).transpose(1, 2)
    anscombe = Anscombe()
    Y_train_anscombed = anscombe.forward(Y_train_tensor)
    Y_train_Y_mean = torch.mean(Y_train_anscombed, dim=0, keepdim=True)

    train_dataset = NeuralDataset(X_position_train, X_dense_task_train, X_sparse_task_train, Y_train, Y_train_Y_mean, device=device)
    valid_dataset = NeuralDataset(X_position_valid, X_dense_task_valid, X_sparse_task_valid, Y_valid, Y_train_Y_mean, device=device)
    
    train_loader = DataLoader(train_dataset, batch_size=Conf.training.batch_size, shuffle=True)
    valid_loader = DataLoader(valid_dataset, batch_size=10000, shuffle=False)

    train_loader_folds.append(train_loader)
    valid_loader_folds.append(valid_loader)


In [ ]:
def folds_trainer(Conf, train_loader, valid_loader, mean_model, cov_model, base_lit_models=None, seq_size_cov=1):
    outer_bar = tqdm(total=len(Conf), desc="Folds", position=0, leave=True, dynamic_ncols=True)

    lit_models = []
    trainers = []
    metrics = []
    for fold_idx, (train_loader, valid_loader) in enumerate(
        zip(train_loader_folds, valid_loader_folds), start=0 
    ):
        trainer, lit_model = build_lit_model(Conf, mean_model, cov_model, fold_idx, len(train_loader), seq_size_cov)

        if base_lit_models is not None:
            base_model_state_dict = base_lit_models[fold_idx].full_model.mean_model.state_dict()
            lit_model.full_model.mean_model.load_state_dict(base_model_state_dict, strict=True)
            for p in lit_model.full_model.mean_model.parameters():
                p.requires_grad = False
    
        trainer.fit(lit_model, train_loader, valid_loader)
        lit_models.append(lit_model)
        trainers.append(trainer)

        metric = {k: float(v.detach().cpu()) for k, v in trainer.callback_metrics.items() if torch.is_tensor(v)}
        metrics.append(metric)
        
        mean_valid_loss = np.mean([metric["valid_loss_epoch"] for metric in metrics if "valid_loss_epoch" in metric])
        mean_valid_corr = np.mean([metric["valid_correlation"] for metric in metrics if "valid_correlation" in metric])
        mean_train_loss = np.mean([metric["train_loss_epoch"] for metric in metrics if "train_loss_epoch" in metric])
        mean_train_corr = np.mean([metric["train_correlation"] for metric in metrics if "train_correlation" in metric])
    
        outer_bar.set_description(f"Fold {fold_idx}/{len(Conf)}")
        outer_bar.set_postfix({
            "val_loss_fold": f"{mean_valid_loss:.3f}",
            "val_corr_fold": f"{mean_valid_corr:.3f}",
            "train_loss_fold": f"{mean_train_loss:.3f}",
            "train_corr_fold": f"{mean_train_corr:.3f}",
        })
        outer_bar.update(1)
        
    return lit_models, trainers

In [ ]:
baseline_lit_models, baseline_trainers = folds_trainer(Conf, train_loader_folds, valid_loader_folds, "baseline", "identity")

In [ ]:
identity_lit_models, identity_trainers = folds_trainer(Conf, train_loader_folds, valid_loader_folds, "conditional", "identity")

In [ ]:
shared_lit_models, shared_trainers = folds_trainer(Conf, train_loader_folds, valid_loader_folds, "conditional", "shared", identity_lit_models)

In [ ]:
temposhared_lit_models, temposhared_trainers = folds_trainer(Conf, train_loader_folds, valid_loader_folds, "conditional", "shared", identity_lit_models, seq_size_cov=13)

In [ ]:
conditional_lit_models, conditional_trainers = folds_trainer(Conf, train_loader_folds, valid_loader_folds, "conditional", "conditional", identity_lit_models)

In [ ]:
tempoconditional_lit_models, tempoconditional_trainers = folds_trainer(Conf, train_loader_folds, valid_loader_folds, "conditional", "conditional", identity_lit_models, seq_size_cov=13)